In [47]:
import pandas as pd

df = pd.read_csv('../data/processed/cardiac_patient_data_clean.csv')

y = df['zgon']
X = df.drop(columns=['zgon', 'Time of observation', 'days_implantation_to_end'])
X.head()

,LVEF,Wskazania,typo urządzenia,producent,ageyrscont,Płeć,ICD_CRT,ile od wszczepienia,Lcewek,MI,...,"TL_ica,ICA,EDV-ICA cm/s","TL_ica,ICA,PSV-ICA cm/s","TR_cca,CCA,EDV-CCA cm/s","TR_cca,CCA,KIM-CCAr","TR_cca,CCA,PI-CCA","TR_cca,CCA,PSV-CCA cm/s","TR_cca,CCA,RI-CCA","TR_ica,ICA,EDV-ICA cm/s","TR_ica,ICA,PSV-ICA cm/s","TL_cca,CCA,FPP"
0,40.76,zawal_pierwotna,ICD_D,sjm,75.45,1,0,2.0,1,1,...,23.8,124.3,15.23,1.2,2.156,81.1,1.2792,12.9,102.3,2.786512
1,38.78,zawal_pierwotna,ICD_D,med,86.05,1,0,24.0,2,1,...,33.6,102.2,12.50,1.1,1.650,55.5,0.7700,23.1,69.4,2.422503
2,18.82,niewydolnosc_pierwotna,crtd2,bio,85.57,1,1,10.0,1,0,...,21.6,48.7,13.90,1.2,1.330,44.0,0.6800,28.7,62.2,2.322680
3,29.99,zawal_pierwotna,ICD_D,med,82.16,1,0,20.0,2,1,...,15.1,41.3,15.30,0.5,1.410,55.7,0.7300,18.2,43.7,2.209063
4,30.92,zawal_pierwotna,ICD_D,med,81.74,1,0,32.0,2,1,...,18.0,47.3,14.00,1.0,1.840,66.5,0.7900,22.3,61.1,2.559221


In [48]:
text_columns = df.select_dtypes(include=['object', 'category', 'string']).columns

for col in text_columns:
    print(f"--- column: {col} ---")
    print(f"number of unique values: {df[col].nunique()}")
    print(f"Examples: {sorted(df[col].dropna().unique()[:5])}")
    print()

--- column: Wskazania ---
number of unique values: 7
Examples: ['niewydolnosc_pierwotna', 'niewydolnosc_wtorna', 'przebyte_vg_vt', 'zawal_pierwotna', 'zawal_wtorna']

--- column: typo urządzenia ---
number of unique values: 6
Examples: ['CHECK', 'CRTD3', 'ICD_D', 'ICD_V', 'crtd2']

--- column: producent ---
number of unique values: 5
Examples: ['0', 'bio', 'bos', 'med', 'sjm']



In [49]:
X = pd.get_dummies(X, drop_first=True, dtype=int)

In [62]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import VarianceThreshold, f_classif, mutual_info_classif, RFE
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# 1. Simple variance thresholding
var_selector = VarianceThreshold(threshold=0.005)
var_selector.fit(X)
X_filtered = X.loc[:, var_selector.get_support()]

# 2. ANOVA F-test / t-Student
f_scores, p_values = f_classif(X_filtered, y)

# 3. Mutual Information
mi_scores = mutual_info_classif(X_filtered, y, random_state=42)

# 4. Random Forest Classifier Importance
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_filtered, y)
rf_importance = rf.feature_importances_

# 5. Recursive Feature Elimination (RFE)
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_filtered)
rfe_model = LogisticRegression(max_iter=1500, random_state=42)
rfe = RFE(estimator=rfe_model, n_features_to_select=1, step=1)
rfe.fit(X_scaled, y)
rfe_ranks = rfe.ranking_

feature_stats = pd.DataFrame({
    'Feature': X_filtered.columns,
    'F_Score (ANOVA)': f_scores,
    'p_value': p_values,
    'Mutual_Info': mi_scores,
    'RF_Importance': rf_importance,
    'RFE_Rank': rfe_ranks
})

statistically_significant = feature_stats[feature_stats['p_value'] < 0.05].sort_values(
    by='F_Score (ANOVA)', 
    ascending=False
)

top_20 = statistically_significant.head(20)

top_20.style \
    .format({
        'F_Score (ANOVA)': '{:.2f}',
        'p_value': '{:.2e}',
        'Mutual_Info': '{:.4f}',
        'RF_Importance': '{:.4f}',
        'RFE_Rank': '{:d}'
    }) \
    .background_gradient(cmap='Blues', subset=['F_Score (ANOVA)']) \
    .background_gradient(cmap='Blues_r', subset=['p_value']) \
    .background_gradient(cmap='Blues', subset=['Mutual_Info']) \
    .background_gradient(cmap='Blues', subset=['RF_Importance']) \
    .background_gradient(cmap='Blues_r', subset=['RFE_Rank']) \
    .hide(axis='index')

Feature,F_Score (ANOVA),p_value,Mutual_Info,RF_Importance,RFE_Rank
B_NT - proBNP,49.26,9.95e-12,0.0988,0.0448,5
Ea'_mean,41.76,3.08e-10,0.0440,0.0146,53
E_diastarea,39.45,8.97e-10,0.0712,0.0154,1
egfr_CG,39.41,9.13e-10,0.0458,0.0150,2
E_LV2j rozk cm2,37.20,2.57e-09,0.0611,0.0099,4
B_GFR wg MDRD,35.28,6.31e-09,0.0977,0.0079,7
MDRDcorr,32.55,2.30e-08,0.0676,0.0144,38
"E_rozk,LAA cm2.1",31.91,3.12e-08,0.0449,0.0097,20
B_Kreatynina,31.34,4.08e-08,0.0732,0.0226,74
"6mwt_,Dystans",30.91,5.02e-08,0.0653,0.0083,3
